In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
from pprint import pprint

def parse_ucheba_page(url):
    print(" Начинаем парсинг страницы...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        universities = []

        # Ищем все карточки вузов по новому классу
        uni_blocks = soup.find_all('section', class_='search-results-item')
        print(f" Найдено карточек вузов: {len(uni_blocks)}")

        for i, block in enumerate(uni_blocks[:10]):  # Ограничимся первыми 10
            try:
                print(f"\n Обработка карточки #{i+1}")

                # Название вуза
                name = block.find('h2', class_='search-results-title').get_text(strip=True)
                name = re.sub(r'\s+', ' ', name)  # Удаляем лишние пробелы
                print(f"🏛 Название: {name}")

                # Проходной балл
                passing_block = block.find('section', class_='sro-point')
                passing_score = np.nan
                if passing_block:
                    passing_text = passing_block.find('div', class_='big-number-h2').get_text(strip=True)
                    passing_score = float(re.sub(r'[^\d.]', '', passing_text))

                # Бюджетные места
                budget_block = block.find('section', class_='sro-place_sum')
                budget_places = np.nan
                if budget_block:
                    budget_text = budget_block.find('div', class_='big-number-h2').get_text(strip=True)
                    budget_places = int(re.sub(r'[^\d]', '', budget_text))

                # Стоимость обучения
                price_block = block.find('section', class_='sro-price')
                price = np.nan
                if price_block:
                    price_text = price_block.find('div', class_='big-number-h2').get_text(strip=True)
                    price = float(re.sub(r'[^\d]', '', price_text))

                universities.append({
                    'Название вуза': name,
                    'Проходной балл': passing_score,
                    'Бюджетных мест': budget_places,
                    'Стоимость обучения': price
                })

                print("📊 Результат:", universities[-1])

            except Exception as e:
                print(f" Ошибка при обработке карточки #{i+1}: {str(e)}")
                continue

        return universities

    except Exception as e:
        print(f"🔥 Критическая ошибка: {str(e)}")
        return []

# URL для тестирования
test_url = "https://web.archive.org/web/20160213233521/https://www.ucheba.ru/for-abiturients/vuz"
print(f"\n Загружаем данные с: {test_url}")
data = parse_ucheba_page(test_url)

if data:
    df = pd.DataFrame(data)

    # Очистка данных
    df = df.drop_duplicates(subset=['Название вуза'])
    df = df.replace(0, np.nan)

    # Сохраняем результаты
    df.to_csv('universities_data_final.csv', index=False, encoding='utf-8-sig')

    print("\n Успешно собрано данных:", len(df))
    print("\nПример данных:")
    pprint(df.head().to_dict('records'))

    print("\n Статистика:")
    print(df.notna().sum())
else:
    print("\n Не удалось собрать данные. Проверьте:")
    print("1. Доступность страницы")
    print("2. Наличие данных в HTML-коде")
    print("3. Классы элементов (возможно изменились)")


🌐 Загружаем данные с: https://web.archive.org/web/20160213233521/https://www.ucheba.ru/for-abiturients/vuz
🚀 Начинаем парсинг страницы...
🎓 Найдено карточек вузов: 20

📊 Обработка карточки #1
🏛 Название: Национальный исследовательский университет «Высшая школа экономики»
📊 Результат: {'Название вуза': 'Национальный исследовательский университет «Высшая школа экономики»', 'Проходной балл': 216.0, 'Бюджетных мест': 1963, 'Стоимость обучения': 240000.0}

📊 Обработка карточки #2
🏛 Название: Московский государственный лингвистический университет
📊 Результат: {'Название вуза': 'Московский государственный лингвистический университет', 'Проходной балл': 160.0, 'Бюджетных мест': 901, 'Стоимость обучения': 86800.0}

📊 Обработка карточки #3
🏛 Название: Московский технический университет связи и информатики
📊 Результат: {'Название вуза': 'Московский технический университет связи и информатики', 'Проходной балл': 137.0, 'Бюджетных мест': 681, 'Стоимость обучения': 57000.0}

📊 Обработка карточки #4

In [ ]:
!pip install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.9/492.9 kB 30.8 MB/s eta 0:00:00


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time


def extract_archive_date(url):
    """Извлекает дату из URL архивной версии"""
    try:
        # Ищем паттерн даты в URL (формат /web/YYYYMMDDHHMMSS/)
        date_str = re.search(r'/web/(\d{4})(\d{2})(\d{2})(\d{2})(\d{2})(\d{2})/', url)
        if date_str:
            year, month, day, hour, minute, second = date_str.groups()
            return f"{day}.{month}.{year} {hour}:{minute}:{second}"
        return "Дата не определена"
    except:
        return "Дата не определена"


def setup_driver():
    """Настройка Selenium WebDriver"""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver

def clean_numeric_value(text):
    """Очистка числовых значений, замена прочерков на 0"""
    if not text or text.strip() == '—':
        return 0
    try:
        return float(re.sub(r'[^\d.]', '', text.strip()))
    except:
        return 0

def parse_program(program_block):
    """Парсинг данных одной программы"""
    data = {
        'Программа': np.nan,
        'Уровень': np.nan,
        'Факультет': np.nan,
        'Проходной балл': 0,
        'Бюджетные места': 0,
        'Стоимость': 0
    }

    try:
        # Название программы
        name_tag = program_block.find('h3', class_='search-results-title')
        if name_tag:
            data['Программа'] = name_tag.get_text(strip=True)

        # Уровень образования
        level_tag = program_block.find('div', class_='fs-small')
        if level_tag:
            data['Уровень'] = level_tag.get_text(strip=True)

        # Факультет
        faculty_tag = program_block.find('h4', class_='search-results-info-big')
        if faculty_tag:
            data['Факультет'] = faculty_tag.get_text(strip=True)

        # Числовые показатели
        options_div = program_block.find('div', class_='row')
        if options_div:
            # Проходной балл
            passing_div = options_div.find('section', class_='sro-point')
            if passing_div:
                score = passing_div.find('div', class_='big-number-h2')
                if score:
                    data['Проходной балл'] = clean_numeric_value(score.get_text())

            # Бюджетные места
            budget_div = options_div.find('section', class_='sro-place')
            if budget_div:
                places = budget_div.find('div', class_='big-number-h2')
                if places:
                    data['Бюджетные места'] = int(clean_numeric_value(places.get_text()))

            # Стоимость
            price_div = options_div.find('section', class_='sro-price_interval')
            if price_div:
                price = price_div.find('div', class_='big-number-h2')
                if price:
                    data['Стоимость'] = clean_numeric_value(price.get_text())

    except Exception as e:
        print(f"Ошибка парсинга программы: {str(e)}")

    return data

def parse_ucheba_page(url):
    print(" Запускаем парсинг с Selenium...")
    driver = setup_driver()
    results = []
    archive_date = extract_archive_date(url)

    try:
        driver.get(url)
        print(f" Ожидаем загрузки элементов (архив от {archive_date})...")

        WebDriverWait(driver, 40).until(
            EC.presence_of_element_located((By.CLASS_NAME, 'search-results-item')))

        uni_blocks = driver.find_elements(By.CLASS_NAME, 'search-results-item')
        print(f" Найдено вузов: {len(uni_blocks)}")

        for i, uni in enumerate(uni_blocks):
            try:
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", uni)
                uni_name = WebDriverWait(uni, 15).until(
                    EC.visibility_of_element_located((By.CLASS_NAME, 'search-results-title'))).text
                uni_name = re.sub(r'\s+', ' ', uni_name).strip()
                print(f"\n🏛 Вуз {i+1}: {uni_name}")

                try:
                    show_programs_btn = WebDriverWait(uni, 15).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, '.js-search-results-more-info, .js-search-results-toggle')))
                    print(f" Найдена кнопка: {show_programs_btn.text}")
                except:
                    print(" Не найдена кнопка раскрытия программ")
                    continue

                # Сохраняем текущее состояние DOM для сравнения
                initial_html = driver.page_source

                # Клик с обработкой возможных ошибок
                try:
                    driver.execute_script("arguments[0].click();", show_programs_btn)
                except Exception as e:
                    print(f" Ошибка при клике: {str(e)}")
                    continue

                try:
                    WebDriverWait(driver, 15).until(
                        lambda d: d.page_source != initial_html)
                    WebDriverWait(driver, 15).until(
                        lambda d: d.find_elements(By.CLASS_NAME, 'search-results-info-item') or
                                not d.find_elements(By.CSS_SELECTOR, '.fa-spin, .search-results-load-icon')
                    )
                    WebDriverWait(driver, 15).until(
                        EC.visibility_of_any_elements_located((By.CLASS_NAME, 'search-results-info-item')))

                except Exception as e:
                    print(f" Ошибка ожидания загрузки: {str(e)}")

                try:
                    programs_html = uni.find_element(
                        By.CSS_SELECTOR, '.search-results-info, .programs-list').get_attribute('outerHTML')

                    if len(programs_html) < 100:  # Если слишком короткий HTML
                        programs_html = driver.find_element(
                            By.CSS_SELECTOR, 'body').get_attribute('outerHTML')

                    soup = BeautifulSoup(programs_html, 'html.parser')


                    programs = soup.find_all('section', class_=lambda x: x and
                                           ('search-results-info-item' in x or
                                            'program-item' in x or
                                            'edu-program' in x))

                    print(f" Найдено программ: {len(programs)}")

                    if len(programs) == 0:
                        print("ℹ Попробуем альтернативный метод поиска...")
                        programs = soup.find_all('div', class_=lambda x: x and
                                               ('program-card' in x or
                                                'edu-program-card' in x))
                        print(f" Найдено программ (альтернативный метод): {len(programs)}")

                    for program in programs:
                        program_data = parse_program(program)
                        program_data.update({
                            'Вуз': uni_name,
                            'Дата архивации': archive_date,
                            'Статус загрузки': 'success' if len(programs) > 0 else 'empty'
                        })
                        results.append(program_data)
                        print(f"    {program_data['Программа']}")

                except Exception as e:
                    print(f" Ошибка парсинга программ: {str(e)}")
                    # Добавляем запись даже при ошибке
                    results.append({
                        'Вуз': uni_name,
                        'Дата архивации': archive_date,
                        'Статус загрузки': f'error: {str(e)}'
                    })

            except Exception as e:
                print(f" Ошибка обработки вуза #{i+1}: {str(e)}")
                continue

        return pd.DataFrame(results)

    finally:
        try:
            driver.quit()
        except:
            pass
# # Запуск парсера
# try:
#     test_url = "https://web.archive.org/web/20160213233521/https://www.ucheba.ru/for-abiturients/vuz"
#     print(f"\n🌐 Загружаем данные с: {test_url}")
#     df = parse_ucheba_page(test_url)

#     if not df.empty:
#         # Дополнительная очистка данных
#         df = df.dropna(subset=['Программа'])

#         # Замена оставшихся NaN (если есть) на 0 для числовых колонок
#         numeric_cols = ['Проходной балл', 'Бюджетные места', 'Стоимость']
#         df[numeric_cols] = df[numeric_cols].fillna(0)

#         # Сохранение
#         df.to_csv('ucheba_programs_final.csv', index=False, encoding='utf-8-sig')

#         print("\n✅ Успешно собрано программ:", len(df))
#         print("\nПример данных:")
#         print(df.head().to_markdown(index=False, tablefmt="grid"))
#     else:
#         print("\n❌ Не удалось собрать данные")
# except Exception as e:
#     print(f"🔥 Критическая ошибка: {str(e)}")

In [ ]:
def setup_driver():
    """Настройка Selenium WebDriver с увеличенными таймаутами"""
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(90)
    driver.implicitly_wait(15)
    return driver

def parse_ucheba_page(url):
    print("🚀 Запускаем улучшенный парсинг...")
    driver = setup_driver()
    results = []
    archive_date = extract_archive_date(url)

    try:
        driver.get(url)
        WebDriverWait(driver, 60).until(
            EC.presence_of_element_located((By.CLASS_NAME, 'search-results-item'))
        )

        # Постоянное обновление списка вузов
        uni_blocks = driver.find_elements(By.CLASS_NAME, 'search-results-item')
        print(f"🎓 Всего вузов: {len(uni_blocks)}")

        for i in range(len(uni_blocks)):
            try:
                # заново находим элементы, так как DOM мог измениться
                current_uni = driver.find_elements(By.CLASS_NAME, 'search-results-item')[i]

                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", current_uni)
                time.sleep(2)

                uni_name = WebDriverWait(current_uni, 25).until(
                    EC.visibility_of_element_located((By.CLASS_NAME, 'search-results-title'))
                ).text.strip()
                print(f"\n🏛 Вуз {i+1}: {uni_name}")

                # Находим кнопку в текущем элементе вуза
                show_programs_btn = WebDriverWait(current_uni, 25).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, '.js-search-results-more-info'))
                )
                print(f"🔄 Найдена кнопка: {show_programs_btn.text}")

                # Клик с восстановлением состояния
                driver.execute_script("arguments[0].click();", show_programs_btn)

                # Ожидание загрузки именно этого блока
                WebDriverWait(driver, 30).until(
                    lambda d: current_uni.find_elements(By.CLASS_NAME, 'search-results-info-item') or
                    "loading" not in current_uni.get_attribute("class")
                )

                # Парсинг внутри текущего блока вуза
                programs_html = current_uni.find_element(
                    By.CSS_SELECTOR, '.search-results-info').get_attribute('outerHTML')
                soup = BeautifulSoup(programs_html, 'html.parser')

                programs = soup.find_all('section', class_='search-results-info-item')
                print(f"📚 Найдено программ: {len(programs)}")

                # Закрытие блока программ перед переходом к следующему вузу
                driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", current_uni)
                driver.execute_script("arguments[0].click();", show_programs_btn)
                time.sleep(1)

                # Обработка программ
                for program in programs:
                    program_data = parse_program(program)
                    program_data.update({
                        'Вуз': uni_name,
                        'Дата архивации': archive_date
                    })
                    results.append(program_data)
                    print(f"   ✅ {program_data['Программа'][:25]}...")

            except Exception as e:
                print(f"⚠️ Ошибка обработки вуза {i+1}: {str(e)}")
                continue

        return pd.DataFrame(results)

    finally:
        driver.quit()

In [ ]:
try:
    test_url = "https://web.archive.org/web/20160213233521/https://www.ucheba.ru/for-abiturients/vuz"
    print(f"\n🌐 Загружаем данные с: {test_url}")
    df = parse_ucheba_page(test_url)

    if not df.empty:
        df = df.dropna(subset=['Программа'])

        # Замена оставшихся NaN (если есть) на 0 для числовых колонок
        numeric_cols = ['Проходной балл', 'Бюджетные места', 'Стоимость']
        df[numeric_cols] = df[numeric_cols].fillna(0)

        # Сохранение
        df.to_csv('ucheba_programs_final.csv', index=False, encoding='utf-8-sig')

        print("\n✅ Успешно собрано программ:", len(df))
        print("\nПример данных:")
        print(df.head().to_markdown(index=False, tablefmt="grid"))
    else:
        print("\n❌ Не удалось собрать данные")
except Exception as e:
    print(f"🔥 Критическая ошибка: {str(e)}")


🌐 Загружаем данные с: https://web.archive.org/web/20160213233521/https://www.ucheba.ru/for-abiturients/vuz
🚀 Запускаем улучшенный парсинг...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
from tqdm import tqdm

def parse_all_universities(csv_path):
    # Читаем CSV с архивными ссылками
    df_urls = pd.read_csv(csv_path)
    df_urls = df_urls[200:300]
    all_results = []

    for url in tqdm(df_urls['archive_url'], desc="Парсинг университетов"):
        try:
            df_page = parse_ucheba_page(url)

            if not df_page.empty:
                all_results.append(df_page)
                print(f"✅ Успешно обработано: {url}")
            else:
                print(f"⚠️ Не удалось обработать: {url}")

        except Exception as e:
            print(f"❌ Ошибка при обработке {url}: {str(e)}")
            continue

    final_df = pd.concat(all_results, ignore_index=True)
    final_df.to_csv('all_universities_data_s_0_0_400.csv', index=False, encoding='utf-8-sig')
    print(f"\n🎉 Готово! Сохранено данных: {len(final_df)} строк")
    return final_df

if __name__ == "__main__":
    csv_path = "/content/university_snapshots.csv"
    parse_all_universities(csv_path)

Парсинг университетов:   0%|          | 0/100 [00:00<?, ?it/s]

🚀 Запускаем улучшенный парсинг...
🎓 Всего вузов: 20

🏛 Вуз 1: Санкт-Петербургский филиал Национального исследовательского университета «Высшая школа экономики»
🔄 Найдена кнопка: 13 программ
📚 Найдено программ: 20
   ✅ Филология...
   ✅ Политология и мировая пол...
   ✅ Экономика. Международный ...
   ✅ Менеджмент. Международный...
   ✅ Медиакоммуникации...
   ✅ Востоковедение...
   ✅ Прикладная математика и и...
   ✅ Прикладной анализ данных ...
   ✅ Аналитика в экономике...
   ✅ Управление бизнесом...
   ✅ Бизнес-информатика...
   ✅ Юриспруденция...
   ✅ Социология и социальная и...
   ✅ Управление и аналитика в ...
   ✅ История...
   ✅ Компьютерные технологии, ...
   ✅ Физика...
   ✅ Тексты, языки и цифровые ...
   ✅ Программирование и инжини...
   ✅ Политология и мировая пол...

🏛 Вуз 2: Санкт-Петербургский государственный электротехнический университет «ЛЭТИ» имени В. И. Ульянова (Ленина)
🔄 Найдена кнопка: 24 программы
📚 Найдено программ: 25
   ✅ Реклама и связи с обществ...
   ✅ П

Парсинг университетов:   1%|          | 1/100 [03:04<5:04:40, 184.66s/it]

   ✅ Юриспруденция...
   ✅ Судебная деятельность...
✅ Успешно обработано: https://web.archive.org/web/20200921091827/https://spb.ucheba.ru/for-abiturients/vuz
🚀 Запускаем улучшенный парсинг...
🎓 Всего вузов: 20

🏛 Вуз 1: Санкт-Петербургский филиал Национального исследовательского университета «Высшая школа экономики»
🔄 Найдена кнопка: 13 программ
📚 Найдено программ: 20
   ✅ Филология...
   ✅ Политология и мировая пол...
   ✅ Экономика. Международный ...
   ✅ Менеджмент. Международный...
   ✅ Медиакоммуникации...
   ✅ Востоковедение...
   ✅ Прикладная математика и и...
   ✅ Прикладной анализ данных ...
   ✅ Аналитика в экономике...
   ✅ Управление бизнесом...
   ✅ Бизнес-информатика...
   ✅ Юриспруденция...
   ✅ Социология и социальная и...
   ✅ Управление и аналитика в ...
   ✅ История...
   ✅ Компьютерные технологии, ...
   ✅ Физика...
   ✅ Тексты, языки и цифровые ...
   ✅ Программирование и инжини...
   ✅ Политология и мировая пол...

🏛 Вуз 2: Санкт-Петербургский государственный эле

Парсинг университетов:   2%|▏         | 2/100 [05:38<4:31:55, 166.48s/it]

...

Парсинг университетов:  99%|█████████▉| 99/100 [7:33:35<05:48, 348.52s/it]

⚠️ Не удалось обработать: https://web.archive.org/web/20211027184703/https://kazan.ucheba.ru/for-abiturients/vuz
🚀 Запускаем улучшенный парсинг...
🎓 Всего вузов: 10

🏛 Вуз 1: Ставропольский филиал МИРЭА — Российского технологического университета
🔄 Найдена кнопка: 12 программ
⚠️ Ошибка обработки вуза 1: Message: stale element reference: stale element not found
  (Session info: chrome=135.0.7049.42); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#stale-element-reference-exception
Stacktrace:
#0 0x5a24c8883e6a <unknown>
#1 0x5a24c8335640 <unknown>
#2 0x5a24c8348c3b <unknown>
#3 0x5a24c83479f2 <unknown>
#4 0x5a24c833cb49 <unknown>
#5 0x5a24c833ad9f <unknown>
#6 0x5a24c833eab8 <unknown>
#7 0x5a24c833eb43 <unknown>
#8 0x5a24c8386585 <unknown>
#9 0x5a24c8386d51 <unknown>
#10 0x5a24c837aa63 <unknown>
#11 0x5a24c83ac77d <unknown>
#12 0x5a24c837a4fa <unknown>
#13 0x5a24c83ac91e <unknown>
#14 0x5a24c83d27b5 <unknown>
#15 0x5

Парсинг университетов: 100%|██████████| 100/100 [7:36:49<00:00, 274.09s/it]

⚠️ Ошибка обработки вуза 10: list index out of range
⚠️ Не удалось обработать: https://web.archive.org/web/20221003061128/https://kazan.ucheba.ru/for-abiturients/vuz



🎉 Готово! Сохранено данных: 15614 строк
